# Baseline: Supervised Encoder Without Pretraining

Prefix-safe supervised DME encoder baseline for low-label comparison against `kaggle_forecast_pipeline.ipynb`. The encoder is trained from random initialization on prefix-only sequences, with train-prefix preprocessing and standardized low-label result artifacts.


In [ ]:
# Cell 1 - Setup
import hashlib
import json
import pathlib
import pickle
import random
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "/kaggle/working/denoising-event-sequences", "--quiet"],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import get_num_feature_dim
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits
from src.models.dme_encoder import DMEEncoder
from src.training.finetune import evaluate_finetune, finetune
from src.utils.logging import MetricsLogger
from src.utils.seed import set_seed

SMOKE_RUN = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", device)


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs" / "baselines" / "supervised_encoder"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "logs"
for p in [OUTPUT_DIR, CKPT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "forecasting": {
        "cut_min_ratio": 0.50,
        "cut_max_ratio": 0.80,
        "min_future_events": 2,
        "cut_seed": 42,
    },
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [1.00] if SMOKE_RUN else [0.05, 0.25, 0.50, 0.75, 1.00]
LABEL_SAMPLING_SEEDS = [42] if SMOKE_RUN else [42, 43, 44, 45, 46]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]
DATALOADER_KWARGS = {"num_workers": 0, "pin_memory": device.type == "cuda"}

if SMOKE_RUN:
    config["training"]["num_epochs_finetune"] = 2
    config["training"]["early_stopping_patience"] = 1
print("Output dir:", OUTPUT_DIR)
print("Cutoff policy:", config["forecasting"])


In [ ]:
# Cell 3 - Load data, prefix-only events, preprocessor, and shared splits
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]

def _forecast_cut_bounds(n_events, cfg):
    f_cfg = cfg.get("forecasting", {})
    min_ratio = float(f_cfg.get("cut_min_ratio", 0.50))
    max_ratio = float(f_cfg.get("cut_max_ratio", 0.80))
    min_future = int(f_cfg.get("min_future_events", 2))
    if n_events < min_future + 1:
        return None
    cut_low = max(1, int(np.floor(n_events * min_ratio)))
    cut_high = min(n_events - min_future, int(np.floor(n_events * max_ratio)))
    if cut_high < cut_low:
        cut_low = max(1, n_events - min_future)
        cut_high = n_events - min_future
    if cut_high < cut_low:
        return None
    return cut_low, cut_high


def _stable_entity_seed(entity_id, base_seed=42):
    digest = hashlib.blake2b(f"{base_seed}:{entity_id}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False)


def deterministic_forecast_cut(entity_id, n_events, cfg, base_seed=None):
    bounds = _forecast_cut_bounds(int(n_events), cfg)
    if bounds is None:
        return None
    cut_low, cut_high = bounds
    if base_seed is None:
        base_seed = int(cfg.get("forecasting", {}).get("cut_seed", 42))
    rng = np.random.default_rng(_stable_entity_seed(entity_id, base_seed))
    return int(rng.integers(cut_low, cut_high + 1))


def build_prefix_events(df_source, entity_ids, cfg):
    entity_set = set(entity_ids)
    working = df_source[df_source[GROUP_COL].isin(entity_set)].copy()
    working = working.sort_values([GROUP_COL, TIME_COL], kind="stable").reset_index(drop=True)
    working["forecast_event_pos"] = working.groupby(GROUP_COL, sort=False).cumcount()
    counts = working.groupby(GROUP_COL, sort=False).size()
    cut_by_entity = {}
    for entity_id in entity_ids:
        if entity_id not in counts.index:
            continue
        cut = deterministic_forecast_cut(entity_id, int(counts.loc[entity_id]), cfg)
        if cut is not None:
            cut_by_entity[entity_id] = cut
    working["forecast_cut"] = working[GROUP_COL].map(cut_by_entity)
    prefix = working[
        working["forecast_cut"].notna()
        & (working["forecast_event_pos"] < working["forecast_cut"])
    ].copy()
    prefix["forecast_cut"] = prefix["forecast_cut"].astype(int)
    valid_entity_ids = [entity_id for entity_id in entity_ids if entity_id in cut_by_entity]
    return prefix.reset_index(drop=True), cut_by_entity, valid_entity_ids


def filter_splits_to_valid(splits_in, valid_entity_ids):
    valid = set(valid_entity_ids)
    filtered = {name: [entity_id for entity_id in ids if entity_id in valid] for name, ids in splits_in.items()}
    if any(len(ids) == 0 for ids in filtered.values()):
        raise ValueError(f"Empty split after prefix filtering: { {k: len(v) for k, v in filtered.items()} }")
    return filtered


def assert_no_future_leakage(prefix_df, cut_by_entity):
    counts = prefix_df.groupby(GROUP_COL, sort=False).size()
    for entity_id, cut in cut_by_entity.items():
        if entity_id not in counts.index:
            continue
        observed = int(counts.loc[entity_id])
        if observed != int(cut):
            raise AssertionError(f"Prefix length mismatch for {entity_id}: {observed} != cut {cut}")
    max_pos = prefix_df.groupby(GROUP_COL, sort=False)["forecast_event_pos"].max()
    bad = [entity_id for entity_id, pos in max_pos.items() if int(pos) >= int(cut_by_entity[entity_id])]
    if bad:
        raise AssertionError(f"Future events leaked into prefix for entities: {bad[:5]}")


df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

raw_splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)
raw_entity_ids = raw_splits["train"] + raw_splits["val"] + raw_splits["test"]
df_prefix, forecast_cut_by_entity, valid_entity_ids = build_prefix_events(df, raw_entity_ids, config)
splits = filter_splits_to_valid(raw_splits, valid_entity_ids)
all_entity_ids = splits["train"] + splits["val"] + splits["test"]
df_model = df_prefix[df_prefix[GROUP_COL].isin(set(all_entity_ids))].copy()
forecast_cut_by_entity = {entity_id: forecast_cut_by_entity[entity_id] for entity_id in all_entity_ids}
assert_no_future_leakage(df_model, forecast_cut_by_entity)

prep = EventPreprocessor(config)
prep.fit(df_model, splits["train"])

vocab_info = {
    "event_type_vocab_size": len(prep.vocab[prep.event_type_col]),
    "cat_vocab_sizes": [len(prep.vocab[c]) for c in prep.categorical_cols],
    "num_num_features": get_num_feature_dim(prep, config),
    "num_classes": 2,
}

with open(CKPT_DIR / "preprocessor.pkl", "wb") as f:
    pickle.dump(prep, f)
(CKPT_DIR / "splits.json").write_text(json.dumps(splits, indent=2, default=str))

print("Raw events:", df.shape)
print("Prefix events:", df_model.shape)
print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("Filtered short entities:", len(raw_entity_ids) - len(all_entity_ids))
print(vocab_info)


In [ ]:
# Cell 4 - Shared sampling, metrics, aggregation, and plotting helpers
def get_entity_labels(df_source, group_col, target_col):
    return df_source.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def json_default(obj):
    if isinstance(obj, pathlib.Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if not np.isfinite(value) else value
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return str(obj)


def aggregate_low_label_runs(run_rows, metric_cols=METRIC_COLS):
    runs_df = pd.DataFrame(run_rows)
    if runs_df.empty:
        return runs_df, []
    agg_rows = []
    for (pipeline, fraction), group in runs_df.groupby(["pipeline", "fraction"], sort=True):
        row = {"pipeline": pipeline, "fraction": float(fraction), "n_seeds": int(group["seed"].nunique())}
        for metric in metric_cols:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(values):
                row[f"{metric}_mean"] = float(values.mean())
                row[f"{metric}_std"] = float(values.std(ddof=0))
        agg_rows.append(row)
    agg_df = pd.DataFrame(agg_rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)
    return agg_df, agg_rows


def build_summary_table(agg_rows):
    rows = []
    for row in agg_rows:
        rows.append(
            {
                "pipeline": row["pipeline"],
                "fraction": row["fraction"],
                "n_seeds": row["n_seeds"],
                "roc_auc": f"{row.get('roc_auc_mean', float('nan')):.4f} +/- {row.get('roc_auc_std', 0.0):.4f}",
                "pr_auc": f"{row.get('pr_auc_mean', float('nan')):.4f} +/- {row.get('pr_auc_std', 0.0):.4f}",
                "macro_f1": f"{row.get('macro_f1_mean', float('nan')):.4f} +/- {row.get('macro_f1_std', 0.0):.4f}",
                "balanced_accuracy": f"{row.get('balanced_accuracy_mean', float('nan')):.4f} +/- {row.get('balanced_accuracy_std', 0.0):.4f}",
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)


def plot_low_label_metrics(agg_df, output_dir, metrics=("roc_auc", "pr_auc", "macro_f1")):
    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_paths = {}
    if agg_df.empty:
        return plot_paths
    for metric in metrics:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        if mean_col not in agg_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        for pipeline, group in agg_df.groupby("pipeline", sort=True):
            group = group.sort_values("fraction")
            x = pd.to_numeric(group["fraction"], errors="coerce")
            y = pd.to_numeric(group[mean_col], errors="coerce")
            yerr = pd.to_numeric(group.get(std_col, 0.0), errors="coerce").fillna(0.0)
            ax.errorbar(x, y, yerr=yerr, marker="o", linewidth=2, capsize=4, label=pipeline)
        ax.set_title(f"{metric} by label fraction")
        ax.set_xlabel("Label fraction")
        ax.set_ylabel(metric)
        ax.set_ylim(0.0, 1.0)
        ax.grid(True, alpha=0.25)
        ax.legend()
        path = plot_dir / f"{metric}_by_fraction.png"
        fig.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        plot_paths[metric] = path
    return plot_paths


def save_low_label_outputs(run_rows, prediction_rows, baseline_name, output_dir, extra_artifacts=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    all_runs_df = pd.DataFrame(run_rows)
    if not all_runs_df.empty:
        all_runs_df = all_runs_df.sort_values(["pipeline", "fraction", "seed"]).reset_index(drop=True)
    all_runs_path = output_dir / "low_label_all_runs.csv"
    all_runs_df.to_csv(all_runs_path, index=False)

    agg_df, agg_rows = aggregate_low_label_runs(run_rows)
    agg_path = output_dir / "low_label_aggregate.csv"
    agg_json_path = output_dir / "low_label_aggregate.json"
    agg_df.to_csv(agg_path, index=False)
    agg_json_path.write_text(json.dumps(agg_rows, indent=2, ensure_ascii=False, default=json_default))

    summary_df = build_summary_table(agg_rows)
    summary_path = output_dir / "low_label_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
    pred_path = output_dir / "test_predictions.csv"
    predictions_df.to_csv(pred_path, index=False)

    plot_paths = plot_low_label_metrics(agg_df, output_dir)
    metrics_payload = {
        "baseline": baseline_name,
        "cutoff_policy": config.get("forecasting", {}),
        "metric_columns": METRIC_COLS,
        "split_sizes": {k: len(v) for k, v in splits.items()},
        "artifacts": {
            "low_label_all_runs": all_runs_path,
            "low_label_aggregate": agg_path,
            "low_label_aggregate_json": agg_json_path,
            "low_label_summary": summary_path,
            "test_predictions": pred_path,
            "plots": plot_paths,
            **(extra_artifacts or {}),
        },
        "aggregate": agg_rows,
    }
    metrics_path = output_dir / "metrics.json"
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False, default=json_default))

    print("Low-label results saved:")
    for path in [all_runs_path, agg_path, agg_json_path, summary_path, pred_path, metrics_path]:
        print(" -", path)
    for path in plot_paths.values():
        print(" -", path)
    print("\nLow-label summary:")
    print(summary_df.to_string(index=False))
    try:
        display(summary_df)
    except NameError:
        pass
    return all_runs_df, agg_df, summary_df, predictions_df, metrics_payload


In [ ]:
# Cell 5 - Shared prefix validation/test loaders and prediction helper
batch_size = config["training"]["batch_size"]
val_loader = DataLoader(
    EventSequenceDataset(df_model, splits["val"], prep, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    **DATALOADER_KWARGS,
)
test_loader = DataLoader(
    EventSequenceDataset(df_model, splits["test"], prep, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    **DATALOADER_KWARGS,
)


def collect_predictions(model, loader):
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            entity_ids = batch["entity_id"]
            labels = batch["label"].cpu().numpy()
            moved = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            probs = torch.softmax(model(moved, mode="finetune")["logits"], dim=-1)[:, 1]
            for entity_id, label, proba in zip(entity_ids, labels, probs.cpu().numpy()):
                rows.append({GROUP_COL: entity_id, "target": int(label), "proba": float(proba)})
    return pd.DataFrame(rows)


In [ ]:
# Cell 6 - Low-label supervised encoder runs and standardized outputs
run_rows = []
prediction_rows = []

for fraction in LABEL_FRACTIONS:
    for seed in LABEL_SAMPLING_SEEDS:
        set_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
        train_loader = DataLoader(
            EventSequenceDataset(df_model, train_ids, prep, config, mode="finetune"),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            **DATALOADER_KWARGS,
        )

        run_id = f"supervised_f{int(fraction * 100):03d}_s{seed}"
        run_dir = CKPT_DIR / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        print(f"fraction={fraction:.2f} seed={seed} train_entities={len(train_ids)}")

        model = DMEEncoder(config, vocab_info).to(device)
        logger = MetricsLogger(str(run_dir / "logs"), run_id)
        logger.log_config(config)
        best_ckpt = finetune(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=config,
            output_dir=str(run_dir),
            device=device,
            logger=logger,
            pretrained_checkpoint=None,
            frozen_encoder=False,
            label_fraction=1.0,
            vocab_info=vocab_info,
        )

        ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        metrics = evaluate_finetune(model, test_loader, num_classes=2, device=device)
        row = {
            "baseline": "supervised_encoder",
            "pipeline": "supervised_encoder",
            "fraction": fraction,
            "seed": seed,
            "train_entities": len(train_ids),
            "best_checkpoint": str(best_ckpt),
            **metrics,
        }
        run_rows.append(row)
        (run_dir / "test_metrics.json").write_text(json.dumps(row, indent=2, ensure_ascii=False, default=json_default))
        print(row)

        preds = collect_predictions(model, test_loader)
        preds["baseline"] = "supervised_encoder"
        preds["pipeline"] = "supervised_encoder"
        preds["fraction"] = fraction
        preds["seed"] = seed
        prediction_rows.append(preds)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

all_runs_df, low_label_agg_df, summary_df, predictions_df, metrics_payload = save_low_label_outputs(
    run_rows,
    prediction_rows,
    baseline_name="supervised_encoder",
    output_dir=OUTPUT_DIR,
    extra_artifacts={"preprocessor": CKPT_DIR / "preprocessor.pkl", "splits": CKPT_DIR / "splits.json"},
)

try:
    display(low_label_agg_df)
except NameError:
    print(low_label_agg_df.to_string(index=False))
